### N-gram 模型构建与最大似然估计（MLE）推导

#### 1. 问题设定：Trigram 模型

一个 trigram (n=3) 语言模型基于 (n-1=2) 阶马尔可夫假设，即下一个词 $w_i$ 的出现概率仅依赖于其前两个词 $(w_{i-2}, w_{i-1})$。我们的目标是根据一个文本语料库 $C$，学习到这组条件概率 $P(w_i | w_{i-2}, w_{i-1})$。

#### 2. 最大似然估计 (Maximum Likelihood Estimation, MLE)

最大似然估计的目标是，找到一组模型参数（即条件概率），使得我们观测到的训练语料库 $C = (w_1, w_2, \dots, w_T)$ 出现的概率最大化。这个概率就是**似然函数** $L$。

##### 步骤 1: 写出似然函数

基于马尔可夫假设，整个语料库的联合概率可以近似为所有 trigram 条件概率的连乘：
$$ L = P(C) \approx \prod_{i=1}^{T} P(w_i | w_{i-2}, w_{i-1}) $$
为了计算方便，我们通常最大化其对数形式，即**对数似然函数** $\log L$，因为连乘可以转换为连加：
$$ \log L = \sum_{i=1}^{T} \log P(w_i | w_{i-2}, w_{i-1}) $$

##### 步骤 2: 将对数似然函数与频率计数关联

原始的对数似然函数是按语料库中的**位置 $i$ (从 1 到 $T$)** 进行求和。我们可以等价地将其转换为按**所有唯一的 trigram 类型 $(u,v,w)$** 进行分组求和。

一个特定的对数概率项 $\log P(w|u,v)$ 在总和中出现的次数，正好是其对应的 trigram $(u,v,w)$ 在语料库中出现的次数，即 $\text{Count}(u,v,w)$。

因此，对数似然函数可以重写为：
$$ \log L = \sum_{(u, v, w)} \text{Count}(u, v, w) \times \log P(w | u, v) $$

##### 步骤 3: 问题的分解

我们观察到，对于一个特定的前缀 $(u,v)$，与之相关的概率 $P(w|u,v)$ 只影响 $\log L$ 中前缀为 $(u,v)$ 的那些项。这意味着我们可以**独立地**为每一个前缀 $(u,v)$ 找到能使 $\log L$ 最大化的概率分布。

问题简化为：对于任意一个固定的前缀 $(u,v)$，求解以下最优化问题。

##### 步骤 4: 使用拉格朗日乘子法求解约束优化问题

**优化问题：**

*   **最大化目标函数 $f$:**
    $$ f = \sum_w \text{Count}(u, v, w) \log P(w | u, v) $$
*   **约束条件 $g$:** （对于给定的 $(u,v)$，所有可能的下一个词 $w$ 的概率之和必须为 1）
    $$ g = \sum_w P(w | u, v) - 1 = 0 $$

**求解过程:**

1.  **构造拉格朗日函数 $\Lambda$**:
    为简化书写，令 $p_w = P(w|u,v)$，$C_{uvw} = \text{Count}(u,v,w)$。
    $$ \Lambda(p_w, \lambda) = f - \lambda g = \sum_w C_{uvw} \log p_w - \lambda \left( \sum_w p_w - 1 \right) $$

2.  **对变量 $p_w$ 求偏导数**:
    对任意一个特定的概率 $p_{w'}$ 求导，并令其等于 0。
    $$ \frac{\partial \Lambda}{\partial p_{w'}} = \frac{C_{uvw'}}{p_{w'}} - \lambda = 0 $$

3.  **求解 $p_{w'}$**:
    从上式可得：
    $$ p_{w'} = \frac{C_{uvw'}}{\lambda} $$
    这个关系对所有可能的下一个词 $w'$ 都成立。

4.  **利用约束条件求解 $\lambda$**:
    将上一步的结果代入约束条件 $\sum_w p_w = 1$：
    $$ \sum_w \frac{C_{uvw}}{\lambda} = 1 $$
    因为 $\lambda$ 是一个与 $w$ 无关的常数，可以提取出来：
    $$ \frac{1}{\lambda} \sum_w C_{uvw} = 1 $$
    求和项 $\sum_w C_{uvw}$ 的含义是“所有以 $(u,v)$ 开头的 trigram 的总数”，这正好就是 bigram $(u,v)$ 出现的总次数，即 $\text{Count}(u,v)$。
    所以，$\sum_w C_{uvw} = \text{Count}(u,v)$。
    代入后得到：
    $$ \frac{1}{\lambda} \text{Count}(u,v) = 1 \implies \lambda = \text{Count}(u,v) $$

5.  **得出最终解**:
    最后，将求出的 $\lambda$ 代回到第 3 步的 $p_w$ 表达式中：
    $$ p_w = \frac{C_{uvw}}{\lambda} = \frac{\text{Count}(u,v,w)}{\text{Count}(u,v)} $$
    换回原始的概率符号，我们就得到了最终的 MLE 估计公式：
    $$ P(w|u,v) = \frac{\text{Count}(u,v,w)}{\text{Count}(u,v)} $$

这个推导严谨地证明了，我们直观上认为的“用频率来估计概率”的计算方法，就是 n-gram 语言模型的最大似然估计。